# DuckDB Integration Example

This notebook demonstrates how to integrate PyIceberg with DuckDB for high-performance analytics on Iceberg tables.

## Overview

DuckDB is an in-process SQL OLAP database management system that can query Iceberg tables directly, providing:
- **High-performance queries** using DuckDB's vectorized execution engine
- **Zero-copy data access** to Parquet files in Iceberg tables
- **Familiar SQL interface** for data analysis
- **Seamless integration** with PyIceberg's Python API

## Use Cases

- **Ad-hoc analytics**: Quick exploratory data analysis
- **Data science**: Use DuckDB's advanced SQL functions
- **Performance testing**: Benchmark query performance
- **ETL pipelines**: Use DuckDB for transformations before writing to Iceberg

In [ ]:
# Import required libraries
import os
import tempfile
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

import pyiceberg
from pyiceberg.catalog import load_catalog

print(f"PyIceberg version: {pyiceberg.__version__}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"PyArrow version: {pa.__version__}")

## Setup: Creating an Iceberg Table

First, we'll create a local Iceberg table with sample data that we can then query with DuckDB.

In [ ]:
# Create a temporary warehouse location
warehouse_path = tempfile.mkdtemp(prefix="iceberg_warehouse_")
print(f"Warehouse location: {warehouse_path}")

In [ ]:
# Configure and load the catalog
catalog = load_catalog(
    "default",
    type="sql",
    uri=f"sqlite:///{warehouse_path}/pyiceberg_catalog.db",
    warehouse=f"file://{warehouse_path}",
)

print("Catalog loaded successfully!")

# Create a namespace
catalog.create_namespace("default")
print(f"Available namespaces: {list(catalog.list_namespaces())}")

In [ ]:
# Create sample sales data
import pyarrow.compute as pc

# Sample sales data
data = {
    "transaction_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "customer_id": [101, 102, 101, 103, 102, 104, 101, 105, 103, 102],
    "product_id": [501, 502, 501, 503, 502, 504, 501, 505, 503, 502],
    "quantity": [2, 1, 3, 1, 2, 1, 4, 2, 1, 3],
    "unit_price": [10.0, 25.0, 10.0, 50.0, 25.0, 100.0, 10.0, 75.0, 50.0, 25.0],
    "transaction_date": ["2024-01-01", "2024-01-02", "2024-01-01", "2024-01-03", 
                        "2024-01-02", "2024-01-04", "2024-01-05", "2024-01-03", 
                        "2024-01-04", "2024-01-05"],
}

# Convert to PyArrow table
df = pa.table(data)

# Add computed column: total_amount
df = df.append_column("total_amount", pc.multiply(df["quantity"], df["unit_price"]))

print("Sample sales data:")
print(df)
print(f"\nSchema: {df.schema}")
print(f"Total rows: {len(df)}")

In [ ]:
# Create an Iceberg table with the schema from our dataframe
table = catalog.create_table(
    "default.sales",
    schema=df.schema,
)

print(f"Created table: {table}")
print(f"Table location: {table.location()}")
print(f"Table schema: {table.schema()}")

In [ ]:
# Append data to the table
table.append(df)
print(f"Rows written to Iceberg table: {len(table.scan().to_arrow())}")

## Querying Iceberg Tables with DuckDB

Now let's query the Iceberg table using DuckDB. DuckDB can read Parquet files directly from the Iceberg table's data directory.

In [ ]:
# Initialize DuckDB connection
con = duckdb.connect()

# Find the data directory for our Iceberg table
# Iceberg stores data in the data/ subdirectory of the table location
data_dir = f"{table.location()}/data"
print(f"Data directory: {data_dir}")

# List the Parquet files in the data directory
parquet_files = []
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith('.parquet'):
            parquet_files.append(os.path.join(root, file))

print(f"Found {len(parquet_files)} Parquet file(s)")
if parquet_files:
    print(f"First file: {parquet_files[0]}")

In [ ]:
# Query the Iceberg table using DuckDB
# DuckDB can read Parquet files using glob patterns
query = f"""
SELECT * 
FROM read_parquet('{data_dir}/**/*.parquet')
ORDER BY transaction_id
"""

result = con.execute(query).fetchdf()
print("Query results from DuckDB:")
print(result)
print(f"\nTotal rows: {len(result)}")

## Advanced DuckDB Queries

Let's perform more complex analytics using DuckDB's SQL capabilities.

In [ ]:
# Aggregation query: Sales by customer
customer_sales = con.execute(f"""
SELECT 
    customer_id,
    COUNT(*) as transaction_count,
    SUM(quantity) as total_quantity,
    SUM(total_amount) as total_revenue,
    AVG(total_amount) as avg_transaction_value
FROM read_parquet('{data_dir}/**/*.parquet')
GROUP BY customer_id
ORDER BY total_revenue DESC
""").fetchdf()

print("Sales by customer:")
print(customer_sales)

In [ ]:
# Window function: Compare each transaction to customer average
transaction_analysis = con.execute(f"""
SELECT 
    transaction_id,
    customer_id,
    total_amount,
    AVG(total_amount) OVER (PARTITION BY customer_id) as customer_avg,
    total_amount - AVG(total_amount) OVER (PARTITION BY customer_id) as difference_from_avg
FROM read_parquet('{data_dir}/**/*.parquet')
ORDER BY customer_id, transaction_id
""").fetchdf()

print("Transaction analysis:")
print(transaction_analysis)

In [ ]:
# Filtered query: High-value transactions
high_value = con.execute(f"""
SELECT 
    transaction_id,
    customer_id,
    product_id,
    quantity,
    unit_price,
    total_amount,
    transaction_date
FROM read_parquet('{data_dir}/**/*.parquet')
WHERE total_amount > 50
ORDER BY total_amount DESC
""").fetchdf()

print("High-value transactions (>$50):")
print(high_value)

## Performance Comparison: PyIceberg vs DuckDB

Let's compare query performance between PyIceberg's native scanning and DuckDB's Parquet reading.

In [ ]:
import time

# Performance test: PyIceberg native scan
start_time = time.time()
pyiceberg_result = table.scan().to_arrow()
pyiceberg_time = time.time() - start_time

print(f"PyIceberg scan time: {pyiceberg_time:.4f} seconds")
print(f"Rows returned: {len(pyiceberg_result)}")

# Performance test: DuckDB query
start_time = time.time()
duckdb_result = con.execute(f"SELECT * FROM read_parquet('{data_dir}/**/*.parquet')").fetchdf()
duckdb_time = time.time() - start_time

print(f"DuckDB query time: {duckdb_time:.4f} seconds")
print(f"Rows returned: {len(duckdb_result)}")

print(f"\nPerformance ratio: {pyiceberg_time/duckdb_time:.2f}x")

## Writing Data from DuckDB to Iceberg

You can also use DuckDB for transformations and then write the results back to Iceberg.

In [ ]:
# Transform data with DuckDB: Calculate customer segments
segmented_data = con.execute(f"""
SELECT 
    customer_id,
    SUM(total_amount) as total_revenue,
    COUNT(*) as transaction_count,
    CASE 
        WHEN SUM(total_amount) > 100 THEN 'High Value'
        WHEN SUM(total_amount) > 50 THEN 'Medium Value'
        ELSE 'Low Value'
    END as customer_segment
FROM read_parquet('{data_dir}/**/*.parquet')
GROUP BY customer_id
ORDER BY total_revenue DESC
""").fetchdf()

print("Customer segments:")
print(segmented_data)

In [ ]:
# Convert DuckDB result to PyArrow and write to Iceberg
segmented_arrow = pa.Table.from_pandas(segmented_data)

# Create a new table for customer segments
segments_table = catalog.create_table(
    "default.customer_segments",
    schema=segmented_arrow.schema,
)

# Write the segmented data
segments_table.append(segmented_arrow)
print(f"Customer segments table created with {len(segments_table.scan().to_arrow())} rows")

## Best Practices

### When to Use DuckDB with Iceberg

**Use DuckDB when:**
- You need fast ad-hoc analytics and exploration
- You want to use advanced SQL functions (window functions, CTEs, etc.)
- You're doing data science and statistical analysis
- You need high-performance aggregations and filtering
- You want to prototype queries before productionizing

**Use PyIceberg directly when:**
- You need transactional guarantees (ACID operations)
- You're doing schema evolution
- You need time travel and versioning
- You're building production data pipelines
- You need integration with Iceberg's advanced features (partitioning, Z-ordering, etc.)

### Performance Tips

1. **Use filtering**: DuckDB excels at predicate pushdown, so filter data early
2. **Leverage columnar format**: Only select the columns you need
3. **Use appropriate file sizes**: Iceberg's file sizing affects DuckDB read performance
4. **Consider partitioning**: Well-partitioned data improves both Iceberg and DuckDB performance
5. **Use DuckDB's extensions**: Take advantage of DuckDB's rich ecosystem

### Integration Patterns

1. **Read-only analytics**: Use DuckDB for fast queries on Iceberg data
2. **ETL workflows**: Transform with DuckDB, write back with PyIceberg
3. **Data science**: Use DuckDB for analysis, PyIceberg for data management
4. **Hybrid approaches**: Use both tools based on the specific task

## Conclusion

This example demonstrated how PyIceberg and DuckDB work together seamlessly:

- **PyIceberg** provides robust table management, ACID transactions, and schema evolution
- **DuckDB** provides high-performance SQL analytics on Iceberg's Parquet files
- **Integration** allows you to leverage the strengths of both tools

The combination is particularly powerful for:
- Data exploration and prototyping
- Data science and analytics workflows
- High-performance analytics on large datasets
- Building modern data lakehouse architectures

## Cleanup

Let's clean up the temporary resources created during this example.

In [ ]:
# Close the DuckDB connection
con.close()

# Clean up temporary warehouse directory
import shutil
try:
    shutil.rmtree(warehouse_path)
    print(f"Cleaned up temporary warehouse: {warehouse_path}")
except Exception as e:
    print(f"Cleanup warning: {e}")

print("Example completed successfully!")